<a href="https://colab.research.google.com/github/Ashwita729/ashwita-codeboosters-2026/blob/main/DAY_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')
print(f'Pandas version: {pd.__version__}')
print(f'Requests version: {requests.__version__}')



All libraries imported successfully!
Pandas version: 2.2.2
Requests version: 2.32.4


In [ ]:
# STEP 2: EXTRACT: Load the messy sales data
raw_df = pd.read_csv('messy_sales_data.csv')

print(f'Raw data loaded: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
print(f'Columns: {raw_df.columns.tolist()}')
raw_df.head()


Raw data loaded: 30 rows, 9 columns
Columns: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
#STEP 3 : Diagnose all data quality problems before fixing

print('='*55)
print( ' DATA QUALITY DIAGNOSIS REPORT')
print('='*55)

#1.Missing Values
print('\n[1] Missing Values per column : ')
print(raw_df.isnull().sum())

#2.Duplicates
print(f'\n[2] DUPLICATE ROWS : {raw_df.duplicated().sum()}')

#3.Data Types
print('\n[3] DATA TYPES : ')
print(raw_df.dtypes)

#4.Unique values in text columns
print('\n[4] UNIQUE CATEGORIES : ', raw_df['category'].unique())
print('\n[4] Sample Customer Name: ',raw_df['customer_name'].dropna().unique()[0:8])
print('\n[4] Sample order_data_values: ', raw_df['order_date'].unique()[:6])

 DATA QUALITY DIAGNOSIS REPORT

[1] Missing Values per column : 
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS : 0

[3] DATA TYPES : 
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES :  ['Electronics' 'Accessories' nan]

[4] Sample Customer Name:  ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']

[4] Sample order_data_values:  ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [ ]:
#To find out missing values in quantity column
print(raw_df['quantity'].isnull().sum())

3


In [ ]:
# STEP 4 : Create a working copy(ETL best practice)

df=raw_df.copy()
print(f'Working copy created : {df.shape}')
print('raw_df is untouched - we can always reset by running df = raw_df.copy()')

Working copy created : (30, 9)
raw_df is untouched - we can always reset by running df = raw_df.copy()


In [ ]:
# STEP 5 : Fix 1 -> Handle Missing Values

print('Before fixing null: ', df.isnull().sum().sum(), 'total missing values')

df['customer_name'].fillna('Unknown Customer', inplace=True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f' Filled missing quantity with median : {median_qty}')
df['category'].fillna('Uncategorized',inplace=True)
df['product'].fillna('Unknown Product',inplace=True)

print('After fixing null: ', df.isnull().sum().sum(), 'total missing values')


Before fixing null:  1 total missing values
 Filled missing quantity with median : 2.0
After fixing null:  0 total missing values


In [ ]:
# STEP 6 : Fix 2 -> Remove Duplicates

print(f'Before duplicates: {len(df)} rows')
print(f'Duplicate rows : {df.duplicated().sum()}')
print(f'Duplicate rows: ')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product']])

df.drop_duplicates(inplace=True)
print(f'After duplicates: {len(df)} rows')
print(f'Rows Removed: {len(raw_df) - len(df)}')




#

Before duplicates: 30 rows
Duplicate rows : 0
Duplicate rows: 
Empty DataFrame
Columns: [order_id, customer_name, product]
Index: []
After duplicates: 30 rows
Rows Removed: 0


In [ ]:
# STEP 7 :

print('sample data before parsing: ')
print(df['order_date'].head(8).tolist())

df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)

nat_count = df['order_date'].isnull().sum()
print(f'\nUnparseable dates(NaT): {nat_count}')

df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')

print('sample data after parsing: ')
print(df[['order_date','year','month','month_name']].head(5))

sample data before parsing: 
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00')]

Unparseable dates(NaT): 2
sample data after parsing: 
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
